In [ ]:
import os


try:
    GOOGLE_API_KEY = ""
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")


In [ ]:
from google.genai import types

deterministic_config = types.GenerateContentConfig(
    temperature=0.0,
    top_p=1.0,
    seed=42
)

In [ ]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search



sense_filtering_agent = Agent(
    name="sense_filtering_agent",
    model="gemini-2.5-flash-lite",
    generate_content_config=deterministic_config,
    instruction="""You are an agent who is expert on word semantics and language ambiguity. You task is to filter out the relevant sense IDs and their meanings for a given ambiguous word.
    1. Refer to existing web dictionaries, wikipedia and lexical resources to identify the possible meanings of the given word.
    2. From provided sense IDs and meanings, filter out only those that are relevant to the given word. If you are not sure about any sense ID whether to include or exclude, then you can always include it.
    If the sense ID is clearly contradicting the meaning of the word, you can exclude it.
    3. Return only suitable sense IDs and their meanings.""",
    tools=[google_search],
    output_key="sense_filtering_agent_results", # Stores the initial draft
)

print("✅ sense_filtering_agent created.")


# 2. Define the Root Pipeline (Initial -> Loop)
root_agent_sf = SequentialAgent(
    name="SenseFilteringPipeline",
    sub_agents=[sense_filtering_agent]
)

print("✅ Pipeline created.")


# 3. Run it
runner_sense_filtering = InMemoryRunner(agent=root_agent_sf)



In [ ]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types



def exit_loop():
    """Call this function ONLY when the examples are valid and verify agent approves as "True", indicating the examples matches the criteria and no more changes are needed."""
    return {"status": "approved", "message": " Exiting refinement loop."}

print("✅ exit_loop function created.")

# 1. Initial Search Agent (The Draft Creator)
webSearch_agent = Agent(
    name="webSearch_agent",
    model="gemini-2.5-flash-lite",
    generate_content_config=deterministic_config,
    instruction="""You are a web Search agent.
    1. Refer to existing web dictionaries, wikipedia and lexical resources to identify the possible examples to match the given word sense.
    2. You need to return 2 illustrative examples for each given word sense from the list.
    3. Provide concise and accurate information based on the search results. 
    4. Do not add additional comments or information. Only sense ID, its meaning and 2 examples are expected.""",
    tools=[google_search],
    output_key="websearch_results", # Stores the initial draft
)

print("✅ webSearch_agent created.")


#2. Verify Agent (The Critic/Gatekeeper)
verify_agent = Agent(
    name="verify_agent",
    model="gemini-2.5-flash-lite", 
    generate_content_config=deterministic_config,   
    instruction="""You are an expert verifier. 
    Review the context: {websearch_results}
    
    Check if EACH sense ID has exactly 2 examples.
    
    - IF the data has 2 examples for each sense and meaningful: You MUST return `TRUE` immediately indicating the task is finished and no more changes are needed.
    - IF the data is missing examples or incorrect: Return the string `FALSE` with a brief explanation of what is missing or incorrect. Be specific about which sense IDs need attention. YOu can use google search to provide missing data.   
    - You should only returen `TRUE` or `FALSE` with a brief explanation. Do not add any additional comments or information.""",
    tools=[google_search],
    output_key="verification_feedback", # Stores the feedback/status
    
)

print("✅ verify_agent created.")

# 3. Correction Agent (The Fixer)
correction_agent = Agent(
    name="correction_agent",
    model="gemini-2.5-flash-lite",
    generate_content_config=deterministic_config,
    # CHANGE HERE: Removed "{verification_feedback}" to prevent KeyError.
    # The agent will automatically see the previous message in the chat history.
    instruction="""You are a Correction Agent.
    
    Current Data: {websearch_results}
    Correction needed based on feedback: {verification_feedback}  
   
    if the correction needed with feedback has indicates "TRUE":
    1. Call the `exit_loop`  function and nothing else. No further action is needed.
    
    If the feedback indicates "FALSE" or missing information:
    1. Focus on the missing examples, Feedback and corrections specifically requested in the feedback.
    2. Rewrite the FULL list of sense IDs, meanings, and examples with the needed correction.""",   
    
    tools=[FunctionTool(exit_loop)], 
    output_key="websearch_results", 
)

print("✅ correction_agent created.")


# 6. Define the Root Pipeline (Initial -> Loop)
iterator_agent = LoopAgent(
    name="SearchAndVerifyPipeline",
    sub_agents=[verify_agent, correction_agent],
    max_iterations=2
)

print("✅ Pipeline created.")

root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[webSearch_agent, iterator_agent],
)

# 7. Run it
runner = InMemoryRunner(agent=root_agent)



In [ ]:
import json
import re

def extract_agent_json(text: str):
    """
    Extracts the JSON object containing websearch_results, 
    verification_feedback, and final_examples from the agent output.
    """

    # 1. First try to find a ```json ... ``` codeblock
    codeblock_pattern = r"```json\s*(\{.*?\})\s*```"
    match = re.search(codeblock_pattern, text, flags=re.DOTALL)

    if match:
        json_str = match.group(1)

    else:
        # 2. Fallback: use a brace-stack parser to extract first valid {...}
        json_str = extract_first_json_with_brace_stack(text)
        if json_str is None:
            raise ValueError("No JSON block found.")

    # 3. Parse JSON safely
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON found: {e}")

    # 4. Validate required fields
    required_keys = ["websearch_results", "verification_feedback", "final_examples"]
    for key in required_keys:
        if key not in data:
            raise ValueError(f"Missing expected key: {key}")

    return data


def extract_first_json_with_brace_stack(text: str):
    """
    Extracts the first balanced {...} block using brace counting,
    allowing nested braces (unlike regex).
    """
    start = text.find("{")
    if start == -1:
        return None

    stack = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            stack += 1
        elif text[i] == "}":
            stack -= 1
            if stack == 0:   # JSON block is complete
                return text[start:i+1]

    return None  # Unbalanced


In [ ]:
from huggingface_hub import login
from datasets import load_dataset
import ast

#Load dataset from Hugging Face
dataset = load_dataset("deshanksuman/42D_sensemapped", split="test")

for line_num, sample in enumerate(dataset, 1):
    sentence = sample["text"]
    word = sample["word"]
    pos = sample["pos"]
    gold = sample["gold"]
    senses = sample["possible_meanings"]
    lst = ast.literal_eval(f"[{senses}]")

       
    prompt_sense_filtering=f'''Given the ambiguous word {word}, and possible meanings {lst},
1. Refer to semantics websites like WordNet, Dictionary, Wikipedia to identify the possible meanings of the given word.
2. From provided sense IDs and meanings, filter out only those that are mostly relevant to the given sentence. The sentence is: "{sentence}"  
3. Return only suitable sense IDs and their meanings.

Output your response in strictly valid JSON format with this structure:
{{  
  "filtered_senseIds": {{
    "SenseID 1": "meaning",
    "SenseID 2": "meaning"
  }}
}}
'''
    response = await runner_sense_filtering.run_debug(prompt_sense_filtering)
    sense_filtering_agent_results=extract_agent_json_senses(str(response))["filtered_senseIds"]


    # Calling the agentic pipeline
    prompt = f"""
The word '{word}' has multiple senses. The possible senses are: {sense_filtering_agent_results}.
You need to find 2 examples for each sense from reliable web sources.
Output your response in strictly valid JSON format with this structure:

{{
  "websearch_results": "summary of what you found",
  "verification_feedback": "True or False with explanation",
  "final_examples": {{
    "SenseID 1": ["example 1", "example 2"],
    "SenseID 2": ["example 1", "example 2"]
  }}
}}
"""
    
    response = await runner.run_debug(prompt)     
    break
       

              

In [ ]:
import json

def save_examples_to_json(data_list, output_file):
    """
    Converts a list of [id, prompt, goldSense] into a JSON list
    and saves it to a file.

    Args:
        data_list (list): List of lists, each containing [id, prompt, goldSense]
        output_file (str): File path to save JSON output
    """

    json_list = []

    for item in data_list:
        if len(item) != 3:
            raise ValueError(f"Expected 3 elements per item, got {len(item)}: {item}")

        example_id, prompt, goldSense = item

        json_list.append({
            "id": example_id,
            "prompt": prompt,
            "goldSense": goldSense
        })

    # Write to file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(json_list, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(json_list)} items to {output_file}")


save_examples_to_json(new_prompt_list, "examples.json")
